# Lab dl_02 · Bigram language model: counting vs. learning

**Goal.** Model words character by character with $P(c_t \mid c_{t-1})$. First by counting (maximum likelihood),
then as a neural network trained by gradient descent — and see that both give the same model.

### Theory recap
A language model factorises $P(c_1\ldots c_n) = \prod_t P(c_t\mid c_{<t})$. A **bigram** model keeps only the previous character.
We wrap every word in a boundary token `.`: `emma` → `.emma.` gives the bigrams `.e, em, mm, ma, a.`.

* **MLE:** $\hat P(b\mid a) = \dfrac{c(a,b)}{\sum_{b'} c(a,b')}$. **Add-$\alpha$ smoothing:** $\dfrac{c(a,b)+\alpha}{c(a)+\alpha V}$.
* **Average NLL** over $N$ bigrams: $\ell = -\frac1N\sum\log \hat P(c_t\mid c_{t-1})$; **perplexity** $= e^{\ell}$.
* **Neural bigram:** logits $= \mathrm{onehot}(a)\,W$ with $W\in\mathbb R^{V\times V}$; softmax of row $a$ of $W$ is the
  predicted distribution. Minimising cross-entropy drives $\mathrm{softmax}(W_a)\to\hat P(\cdot\mid a)$ — so $W$ plays the role of log-counts.
  An L2 penalty on $W$ acts like smoothing (pulls rows toward uniform).

Data: all distinct lowercase words in TinyShakespeare (`data/tinyshakespeare.txt`).

**How this works.** Fill in every cell marked `# TODO` (they currently `raise NotImplementedError`). Run the notebook top to bottom with the working directory set to `labs/`. The last cell runs hidden checks on your functions and records the result in the Learning Center. Do the paper homework for the unlocking skill first — the lab assumes you can do the math by hand.

*This lab is original course material, inspired by Andrej Karpathy's open-source teaching code ([micrograd](https://github.com/karpathy/micrograd), [nn-zero-to-hero](https://github.com/karpathy/nn-zero-to-hero), [nanoGPT](https://github.com/karpathy/nanoGPT), [build-nanogpt](https://github.com/karpathy/build-nanogpt), [minbpe](https://github.com/karpathy/minbpe)). Watching the corresponding lecture after finishing is highly recommended.*

In [ ]:
import math
import re
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

BOS = "."

def load_words(path="data/tinyshakespeare.txt", min_len=2):
    text = Path(path).read_text(encoding="utf-8").lower()
    return sorted({w for w in re.findall(r"[a-z]+", text) if len(w) >= min_len})

words = load_words()
print(len(words), "words, e.g.", words[:8])

## 1. Vocabulary and counts
`build_vocab(words)` → `(stoi, itos)` with `.` at index 0 and the letters at 1, 2, … in sorted order.
`bigram_counts(words, stoi)` → an integer tensor `N` of shape `(V, V)`, `N[i, j]` = count of bigram (itos[i], itos[j]).

In [ ]:
def build_vocab(words):
    # TODO
    raise NotImplementedError


def bigram_counts(words, stoi):
    # TODO
    raise NotImplementedError

In [ ]:
stoi, itos = build_vocab(words)
N = bigram_counts(words, stoi)
V = len(stoi)
plt.figure(figsize=(8, 8)); plt.imshow(N, cmap="Blues")
plt.xticks(range(V), [itos[i] for i in range(V)]); plt.yticks(range(V), [itos[i] for i in range(V)])
plt.title("bigram counts (row = previous char)"); plt.show()

## 2. Probabilities, NLL, perplexity, sampling

In [ ]:
def counts_to_probs(N, alpha=1.0):
    # TODO: add-alpha smoothing, rows sum to 1 (return a float tensor)
    raise NotImplementedError


def nll(words, P, stoi):
    # TODO: AVERAGE negative log-likelihood (natural log) over all bigrams of all words
    raise NotImplementedError


def perplexity(avg_nll):
    # TODO
    raise NotImplementedError


def sample_word(P, itos, generator=None, max_len=30):
    # TODO: start from '.', repeatedly sample the next char with torch.multinomial until '.' is drawn
    raise NotImplementedError

In [ ]:
import random
random.Random(0).shuffle(words)
train_w, val_w = words[: int(0.9 * len(words))], words[int(0.9 * len(words)):]
Ntr = bigram_counts(train_w, stoi)
for alpha in [0.01, 0.1, 1.0, 10.0]:
    P = counts_to_probs(Ntr, alpha)
    print(f"alpha={alpha:5}: train NLL {nll(train_w, P, stoi):.4f}  val NLL {nll(val_w, P, stoi):.4f}  val PPL {perplexity(nll(val_w, P, stoi)):.2f}")
print("uniform model NLL:", math.log(V))
g = torch.Generator().manual_seed(2147483647)
print([sample_word(counts_to_probs(Ntr, 1.0), itos, g) for _ in range(10)])

## 3. The same model as a neural network
`make_bigram_dataset(words, stoi)` → `(xs, ys)` LongTensors of previous / next indices.
`train_neural_bigram(xs, ys, V, steps, lr, reg, seed)` initialises `W = torch.randn((V, V), generator=torch.Generator().manual_seed(seed), requires_grad=True)`, runs **full-batch** gradient descent on `F.cross_entropy(onehot(xs) @ W, ys) + reg * (W**2).mean()` and returns `(W.detach(), losses)`.

In [ ]:
def make_bigram_dataset(words, stoi):
    # TODO
    raise NotImplementedError


def train_neural_bigram(xs, ys, V, steps=200, lr=50.0, reg=0.0, seed=0):
    # TODO
    raise NotImplementedError

In [ ]:
xs, ys = make_bigram_dataset(train_w, stoi)
W, losses = train_neural_bigram(xs, ys, V, steps=300, lr=50.0, reg=0.0)
plt.plot(losses); plt.xlabel("step"); plt.ylabel("train NLL"); plt.show()
P_count = counts_to_probs(Ntr, 0.01)
P_nn = F.softmax(W, dim=1).double()
print("count-model train NLL:", nll(train_w, P_count, stoi), " neural:", losses[-1])
print("max |P_nn - P_count| on frequent rows:", (P_nn - P_count)[Ntr.sum(1) > 1000].abs().max().item())

### Reflect (on paper)
1. Show that minimising the cross-entropy over $W$ (no regularisation) is solved by $\mathrm{softmax}(W_a) = \hat P_{MLE}(\cdot\mid a)$.
2. Which value of $\alpha$ gave the best validation NLL? Explain the trade-off.
3. Why is the validation perplexity of a bigram model on this data still large? What information is it missing?

## Grade
Runs the hidden checks (fast, small inputs, fixed seeds) and reports to the local app.

In [ ]:
from lc_client import grade
grade("dl_02_bigram", globals())